# Earth Engine gridMET to SPI and SPEI

Read daily precipitation and reference evapotranspiration from Google Earth
Engine with Xee, aggregate them to calendar months, and compute SPI and SPEI
with the xarray API.

Set `EE_PROJECT` to a Google Cloud project registered for Earth Engine and
authenticate once with `earthengine authenticate`. The setup, the quotas, and
the terms of use are in `docs/earthengine-climate-engine.md`.

In [ ]:
import os
import sys
from pathlib import Path

import ee
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from xee import helpers

from climate_indices import spei, spi
from climate_indices.drought.indices import Distribution

# The reusable helpers live in scripts/ at the repository root.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
from scripts.earthengine_climate_engine_pilot import crop_grid_to_bbox, month_ends, monthly_server_side

project = os.environ.get("EE_PROJECT", "").strip()
assert project, "set EE_PROJECT to a Google Cloud project registered for Earth Engine"
ee.Initialize(project=project, url="https://earthengine-highvolume.googleapis.com")
print("Earth Engine ready for project:", project)

## Open the collection on a cropped native grid

The native grid covers the whole conterminous United States. Cropping it to the
area of interest keeps the source pixel edges and turns roughly a million cells
into a few thousand.

In [ ]:
BBOX = (-85.7, 29.5, -83.3, 35.0)  # Apalachicola-Chattahoochee-Flint basin, degrees
YEARS = (1980, 2024)

collection = (
    ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
    .filterDate(f"{YEARS[0]}-01-01", f"{YEARS[1]}-12-31")
    .filterBounds(ee.Geometry.Rectangle(BBOX))
    .select(["pr", "eto"])
)
grid = crop_grid_to_bbox(dict(helpers.extract_grid_params(collection)), BBOX)
print("native grid:", helpers.extract_grid_params(collection)["shape_2d"])
print("cropped grid:", grid["shape_2d"])

## Aggregate to calendar months on the server

Forty-five years of daily data is about 16,000 images. Summing each calendar month
inside Earth Engine turns that into 540 images, which is both the cadence the
index needs and a fraction of the requests.

In [ ]:
months = month_ends(*YEARS)
dataset = xr.open_dataset(monthly_server_side(ee, collection, months), engine="ee", **grid)
dataset = dataset.assign_coords(time=("time", [month.last_day for month in months]))
dataset = dataset.rename({"pr": "precipitation", "eto": "reference_et"})

assert dataset.sizes["time"] == len(months), "one step per calendar month, no gaps"
print(dataset)

## Check the units before trusting a value

gridMET publishes both bands in millimetres per day, so a calendar-month sum is
millimetres per month, which is what the index functions take. Earth Engine does
not carry band units into the xarray attributes, so record them here and assert
that the magnitudes are plausible. A metres-versus-millimetres mistake would be
three orders of magnitude low.

In [ ]:
dataset["precipitation"].attrs.update(
    units="mm", long_name="precipitation total", standard_name="precipitation_amount", cell_methods="time: sum"
)
dataset["reference_et"].attrs.update(
    units="mm",
    long_name="ASCE grass reference evapotranspiration total",
    standard_name="water_potential_evapotranspiration_amount",
    cell_methods="time: sum",
)

for name in ("precipitation", "reference_et"):
    values = dataset[name].values
    mean = float(np.nanmean(values))
    assert 1.0 < mean < 500.0, f"{name} monthly mean {mean:.1f} mm is not plausible; check the source unit"
    print(f"{name:14s} mean {mean:6.1f} mm/month   min {np.nanmin(values):6.1f}   max {np.nanmax(values):6.1f}")

## Compute SPI and SPEI

SPI is fitted with a gamma distribution, the conventional choice for
precipitation. SPEI is fitted with the log-logistic distribution, and its input
is precipitation and reference evapotranspiration as separate series:
`spei()` forms the water balance internally, so a dry month keeps its negative
sign.

The calibration window here is the record itself, which is short for a real
study. The cross-validation pilot uses the 1981-2016 baseline that the Earth
Engine drought collection documents. See
`docs/research/earthengine-climate-engine-pilot.md`.

In [ ]:
spi_values = spi(
    dataset["precipitation"],
    3,
    Distribution.gamma,
    calibration_year_initial=YEARS[0],
    calibration_year_final=YEARS[1],
)
spei_values = spei(
    dataset["precipitation"],
    dataset["reference_et"],
    3,
    Distribution.loglogistic,
    calibration_year_initial=YEARS[0],
    calibration_year_final=YEARS[1],
)

for name, values in (("SPI-3", spi_values), ("SPEI-3", spei_values)):
    finite = values.values[np.isfinite(values.values)]
    print(
        f"{name}: {values.attrs['long_name']}, units {values.attrs['units']}, "
        f"{finite.size} finite values, range {finite.min():.2f} to {finite.max():.2f}"
    )

## Plot one grid cell and the latest month

The metadata set by the xarray adapter carries through to the plot labels.

In [ ]:
iy, ix = dataset.sizes["y"] // 2, dataset.sizes["x"] // 2
figure, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
spi_values.isel(y=iy, x=ix).plot(ax=axes[0])
axes[0].set_title("SPI-3 at one grid cell")
axes[0].axhline(0.0, color="black", linewidth=0.8)
spei_values.isel(y=iy, x=ix).plot(ax=axes[1], color="darkorange")
axes[1].set_title("SPEI-3 at the same grid cell")
axes[1].axhline(0.0, color="black", linewidth=0.8)
figure.tight_layout()

In [ ]:
figure, axis = plt.subplots(figsize=(6, 4.5))
spi_values.isel(time=-1).plot(ax=axis, cmap="BrBG", vmin=-2.5, vmax=2.5, cbar_kwargs={"label": "SPI-3"})
axis.set_title(f"SPI-3 for {str(spi_values['time'].values[-1])[:10]}")
axis.set_xlabel("longitude")
axis.set_ylabel("latitude")
figure.tight_layout()

## What to watch for

- **A monthly step that is missing or duplicated** changes every downstream
  value. The assertion above fails loudly instead of shifting the series.
- **The distribution changes the tails.** A rank-based fit cannot express a
  value beyond the range its sample allows; gamma and log-logistic can. The
  pilot quantifies that difference.
- **The calibration window is a choice, not a default.** Fitting on the whole
  record makes the most recent months in-sample.